# Prerequisites

## Load Bronze data to dataframe

In [0]:
%sql
use catalog ecommerce;
create schema if not exists silver;


In [0]:
inventory_df = spark.read.table("bronze.inventory")
display(inventory_df.count())
inventory_df.limit(5).display()

In [0]:
inventory_df.printSchema()

# Transformations

## Drop Duplicates

In [0]:
from pyspark.sql.functions import *

inventory_df = inventory_df.dropDuplicates()
display(inventory_df.count())
inventory_df.limit(5).display()

## Calculate: 
available_to_sell = available_quantity - reserved_quantity

In [0]:
inventory_df = inventory_df.withColumn("available_to_sell", col("available_quantity") - col("reserved_quantity"))
display(inventory_df.count())
inventory_df.limit(5).display()

## Then classify: Stock Status

In [0]:
inventory_df = inventory_df.withColumn(
    "stock_status",
    when(col("available_to_sell") <= 0, "OUT OF STOCK")
    .when(col("available_to_sell") <= col("reorder_level"), "LOW STOCK")
    .otherwise("IN STOCK"),
)
display(inventory_df.count())
inventory_df.limit(5).display()

# Create Silver table

## Write to silver schema

In [0]:
inventory_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.inventory")

## Read silver.inventory table

In [0]:
spark.read.table("ecommerce.silver.inventory").display()